<a href="https://colab.research.google.com/github/RENSHUZHE/HKU-DQMC/blob/main/DQMC_Theory_1_Hubbard_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DQMC Starting with the Hubbard Model

### Statistical Mechanics

For classical models, we have the partition function:

$$Z=\sum_{\sigma} e^{-\beta E_{\sigma}}$$

Physical observables:

$$\langle O\rangle=\frac{1}{Z} \sum_{C} O e^{-\beta E_{C}}$$

For quantum systems, the partition function is:

$$Z=\operatorname{Tr}\left\{e^{-\beta H}\right\}$$

Choosing a basis:

$$Z=\sum_{\{\phi\}}\left\langle\{\phi\}\left|e^{-\beta H}\right|\{\phi\}\right\rangle$$

Physical observables:

$$\langle O\rangle=\frac{\operatorname{Tr}\left\{O e^{-\beta H}\right\}}{Z}$$

### Trotter Decomposition
Considering a system with hamiltonian, $H = H_0 + H_I$, where $H_0$ is the free Hamiltonian, and $H_I$ is the interaction Hamiltonian.

$$\begin{aligned}Z &=\operatorname{Tr}\left[e^{-\beta H}\right] \\&=\operatorname{Tr}\left[(e^{-\Delta \tau H_I} e^{-\Delta \tau H_0})^{L_{\tau}} \right] \\&=\operatorname{Tr}\left[e^{-\Delta \tau H_I} e^{-\Delta \tau H_0} \ldots e^{-\Delta \tau H_I} e^{-\Delta \tau H_0}\right] +\mathcal{O}\left[(\Delta \tau)^{2}\right]\end{aligned}$$

where $L_\tau =\frac{\beta}{\Delta \tau}$

## Why is it called DQMC?

DQMC is mainly used to calculate finite-temperature fermion systems. In the actual operational process, there is only the two-fermion part. Then, for Hamiltonians containing four-fermion interactions like the Hubbard Model, we usually adopt the Hubbard-Stratonovich transformation method, introducing an auxiliary field to decouple it. Then it becomes a two-fermion form, with an additional auxiliary field. (Later we will know that different decoupling methods may lead to the presence or absence of the sign problem).

The reason why DQMC is called Determinantal Monte Carlo is that it converts the calculation of the Trace into calculating the determinant of a matrix. For example, if we want to calculate a Trace in the following form:

$$\operatorname{Tr}\left[e^{-\sum_{i, j} \hat{c}_{i}^{\dagger} A_{i, j} \hat{c}_{j}}\right]=\operatorname{Det}\left[\mathbf{1}+e^{-\mathbf{A}}\right]$$

Where $A_{i, j}$ are the elements of matrix $A$. We can see that the operation of calculating the Trace is converted into calculating the determinant of an identity matrix $\mathbf{1}$ plus the matrix $e^{-\mathbf{A}}$.

There are various ways to prove this, for instance, we can diagonalise the part on the exponent of e, and the Trace is simply the sum over all possibilities for $n_{i}=0,1$:

$$\begin{aligned}&\operatorname{Tr}\left[e^{-\sum_{i, j} \hat{c}_{i}^{\dagger} A_{i, j} \hat{c}_{j}}\right]\\ =&\operatorname{Tr}\left[e^{-\sum_{k} \epsilon(k) \hat{c}_{k}^{\dagger} \hat{c}_{k}}\right]\\ =&\Pi_{k} \operatorname{Tr}\left[e^{-\epsilon(k) \hat{c}_{k}^{\dagger} \hat{c}_{k}}\right]\\ =&\Pi_{k}\left[1+e^{-\epsilon(k)}\right]\end{aligned}$$

The determinant is just the product of eigenvalues, so the above equation obviously holds.

Alternatively, if you feel that having operators on the exponent of e is not direct enough, you can expand it:

$$\exp \left(-c_{k}^{\dagger} \epsilon(k) c_{k}\right)=1-c_{k}^{\dagger} \epsilon(k) c_{k}+\frac{1}{2 !} c_{k}^{\dagger} \epsilon(k) c_{k} c_{k}^{\dagger} \epsilon(k) c_{k}-\frac{1}{3 !} \cdots$$

By further utilising $c_{k}^{\dagger} c_{k}+c_{k} c_{k}^{\dagger}=1$ and $c_{k} c_{k}|\psi\rangle=0$, we have:

$$\begin{aligned}
\exp \left(-c_{k}^{\dagger} \epsilon(k) c_{k}\right) &=1-c_{k}^{\dagger} \epsilon(k) c_{k}+\frac{1}{2 !} c_{k}^{\dagger} \epsilon(k)^{2} c_{k}-\frac{1}{3 !} \cdots \\
&=1+\left[e^{-\epsilon(k)}-1\right] c_{k}^{\dagger} c_{k}
\end{aligned}$$

(Smart students obviously already know that for fermions, $\hat{n}_{k}^{m}=\hat{n}_{k}$...)

And what is important for DQMC is the generalised form of the above equation. For the new form, we can still perform diagonalisation ([PhysRevB.31.4403](https://journals.aps.org/prb/abstract/10.1103/PhysRevB.31.4403)):

$$e^{-\sum_{i, j} c_{i}^{\dagger} A_{i, j} c_{j}} e^{-\sum_{i, j} c_{i}^{\dagger} B_{i, j} c_{j}}=e^{-\sum_{\nu} c_{\nu}^{\dagger} l_{\nu} c_{\nu}}$$

Starting from a single-particle state: $e^{-A} e^{-B}$ acts on $|\psi\rangle=c_{\nu}^{\dagger}|0\rangle$:

$$e^{-\sum_{i, j} c_{i}^{\dagger} A_{i, j} c_{j}} e^{-\sum_{i, j} c_{i}^{\dagger} B_{i, j} c_{j}}|\psi\rangle=\left(e^{-A} e^{-B}\right)_{\nu \nu} c_{\nu}^{\dagger}|0\rangle=e^{-l_{\nu}} c_{\nu}^{\dagger}|0\rangle$$

Considering a many-particle state, for example, a two-particle state $|\phi\rangle=c_{\mu_{1}}^{\dagger} c_{\mu_{2}}^{\dagger}|0\rangle$, we have:

$$\begin{aligned}
e^{-c_{i}^{\dagger} B_{i j} c_{j}}|\phi\rangle &=\prod_{\mu}\left[1+\left(e^{-B_{\mu}}-1\right) c_{\mu}^{\dagger} c_{\mu}\right] c_{\mu_{1}}^{\dagger} c_{\mu_{2}}^{\dagger}|0\rangle \\
&=e^{-B_{\mu_{1}}} e^{-B_{\mu_{2}}} c_{\mu_{1}}^{\dagger} c_{\mu_{2}}^{\dagger}|0\rangle
\end{aligned}$$

Therefore, we have:

$$\operatorname{Tr}\left[e^{-\sum_{i, j} c_{i}^{\dagger} A_{i, j} c_{j}} e^{-\sum_{i, j} c_{i}^{\dagger} B_{i, j} c_{j}}\right]=\operatorname{Det}\left(1+e^{-\mathbf{A}} e^{-\mathbf{B}}\right)$$

For more general cases, the form is self-evident.



## Hubbard Model and Hubbard-Stratonovitch Transformation

We know that for a free fermion system, we can analytically and directly calculate information such as the Matsubara Green's function:

$$G\left(i \omega_{n}\right)=\frac{1}{i \omega_{n}-\left(\epsilon_{k}-\mu\right)}$$

However, for models containing four-fermion interactions like the Hubbard Model, we need a discrete Hubbard-Stratonovitch decomposition, introducing an auxiliary field to carry out Monte Carlo solving.

Historically, this type of DQMC was first used to solve boson-fermion coupling models, also known as the Blankenbecler-Scalapino-Sugar (BSS) algorithm. Later, in 1983, Hirsch proposed the HS transformation for the Hubbard model with on-site Coulomb interactions, that is, by introducing an auxiliary field, decoupling the interaction into the coupling of non-interacting fermions and the auxiliary field, which gradually enabled simple DQMC calculations for interacting fermion systems.

The Hamiltonian of the Hubbard Model is written as:

$$\hat{H}=-t \sum_{\langle i j\rangle \sigma} \hat{c}_{i \sigma}^{\dagger} \hat{c}_{j \sigma}+h.c.+U \sum_{i}\left(\hat{n}_{i \uparrow}-\frac{1}{2}\right)\left(\hat{n}_{i \downarrow}-\frac{1}{2}\right)$$

We usually divide the imaginary time into $M$ slices: $\beta=M \Delta \tau$. Then we perform a Trotter decomposition. The reason for doing this is because $\hat{H}_{0}=-t \sum_{\langle i j\rangle \sigma} \hat{c}_{i \sigma}^{\dagger} \hat{c}_{j \sigma}+h.c.$ and $\hat{H}_{I}=U \sum_{i}\left(\hat{n}_{i \uparrow}-\frac{1}{2}\right)\left(\hat{n}_{i \downarrow}-\frac{1}{2}\right)$ do not commute.

We want to perform the HS transformation on $e^{-\Delta \tau \hat{H}_{I}}$, so we need to separate it out:

$$e^{-\Delta \tau \hat{H}}=e^{-\Delta \tau\left(\hat{H}_{0}+\hat{H}_{I}\right)}=e^{-\Delta \tau \hat{H}_{0}} e^{-\Delta \tau \hat{H}_{I}}+\mathcal{O}\left[(\Delta \tau)^{2}\right]$$

As for why the error is $\mathcal{O}\left[(\Delta \tau)^{2}\right]$, there are multiple ways to understand it, for example, starting from the BCH formula. I will not say more about it here; it is left as a QM exercise.

So what exactly is the HS transformation? In a sense, it is an inverse Gaussian integral.

In our freshman thermodynamics class, we should have learned the formula for the Gaussian integral:

$$e^{A^{2} / 2}=\frac{1}{\sqrt{2 \pi}} \int_{-\infty}^{+\infty} \mathrm{d}\phi e^{-\frac{\phi^{2}}{2}-\phi A}$$

And at this point we know that the Hubbard U term can be written in a squared form:

$$H_{U}=-\frac{U}{2}\left(n_{\uparrow}-n_{\downarrow}\right)^{2}+\frac{U}{4}$$

Then, we simply substitute $A^{2}=\Delta \tau U\left(n_{\uparrow}-n_{\downarrow}\right)^{2}$ into the previous Gaussian integral formula.

In this way, we find that by introducing $\phi$, we have changed the form of $A^{2}$ into the form of $A \phi$, which is a linear term of $A$.

Interestingly, for the Hubbard model, we have a more clever way to decouple:

$$e^{-\Delta \tau H_{U}}=\gamma \sum_{s=\pm 1} e^{\alpha s\left(n_{\uparrow}-n_{\downarrow}\right)}$$

Where $\gamma=\frac{1}{2} e^{-\Delta \tau U / 4}, \quad \cosh(\alpha)=e^{\Delta \tau U / 2}$

The coefficients can be obtained by considering $e^{-\Delta \tau H_{U}}$ in the Hilbert space of a single lattice site:

$$\begin{aligned}
e^{-\Delta \tau U / 4}|0\rangle &=2\gamma|0\rangle \\
e^{-\Delta \tau U / 4}|\uparrow\downarrow\rangle &=2\gamma|\uparrow\downarrow\rangle \\
e^{\Delta \tau U / 4}|\uparrow\rangle &=2\gamma\cosh(\alpha)|\uparrow\rangle \\
e^{\Delta \tau U / 4}|\downarrow\rangle &=2\gamma\cosh(\alpha)|\downarrow\rangle
\end{aligned}$$

This method breaks the $SU(2)$ symmetry of the spin, and we could alternatively choose:

$$e^{-\Delta \tau H_{U}}=\tilde{\gamma} \sum_{s=\pm 1} e^{i\tilde{\alpha}s\left(n_{\uparrow}+n_{\downarrow}-1\right)}$$

Where $\cos(\tilde{\alpha})=e^{-\Delta \tau U/2}, \quad \tilde{\gamma}=\frac{1}{2}e^{\Delta \tau U/4}$. The cost is the introduction of complex numbers.

The situation of the Hubbard Model here is quite special, which is why it can be decoupled by introducing an auxiliary field of $\pm 1$. For more general four-fermion interactions (requiring hermiticity), we have more general HS transformation methods, such as introducing auxiliary fields of $\pm 1$, $\pm 2$:

$$e^{\Delta \tau\lambda A^{2}}=\sum_{l=\pm 1,\pm 2} \frac{\gamma(l)}{4} e^{\sqrt{\Delta \tau\lambda}\eta(l)O} +\mathcal{O}\left(\Delta\tau^{4}\right)$$

where

$$\begin{aligned}
\gamma(\pm 1)=1+\sqrt{6}/3, & \qquad \gamma(\pm 2)=1-\sqrt{6}/3 \\
\eta(\pm 1)=\pm\sqrt{2(3-\sqrt{6})}, &\qquad \eta(\pm 2)=\pm\sqrt{2(3+\sqrt{6})}
\end{aligned}$$

This method is not exact and contains an error related to $\Delta\tau$. For observables, the resulting error is proportional to $\Delta\tau^{3}$.

And how are these current coefficients solved for? I feel that everywhere they just directly give the results, and rarely does anyone manually explain where they come from. Here we might as well assume:

$$\gamma(1)=\gamma(-1)=a, \quad \gamma(2)=\gamma(-2)=b, \quad \eta(1)=\sqrt{c}=-\eta(-1), \quad \eta(2)=\sqrt{d}=-\eta(-2)$$

Then we Taylor expand both sides up to $\mathcal{O}\left(p^{4}\right)$, giving:

$$\begin{array}{c}
1=\frac{1}{2}(a+b) \\
1=\frac{1}{4}(ac+bd) \\
\frac{1}{2}=\frac{1}{48}\left(ac^{2}+bd^{2}\right) \\
\frac{1}{6}=\frac{1}{1440}\left(ac^{3}+bd^{3}\right)
\end{array}$$

Solving this yields:

$$\begin{array}{ll}
a=1+\sqrt{6}/3, & b=1-\sqrt{6}/3 \\
c=2(3-\sqrt{6}), & d=2(3+\sqrt{6})
\end{array}$$

(We will know later that different decoupling methods may lead to the presence or absence of the sign problem. The so-called sign problem here means that for a certain configuration of the auxiliary field, the corresponding weight is a complex or negative number.)

The next section will derive how to calculate the corresponding Weight for a specific configuration of the auxiliary field after introducing the auxiliary field.



## The Weights of the Hubbard Model

We already know from the previous text that when $H=H_{0}+H_{I}$ and usually $[H_{0}, H_{I}] \neq 0$, we similarly divide the imaginary time into $L_{\tau}$ slices: $\beta=L_{\tau}\Delta\tau$, and perform the Trotter decomposition:

$$\begin{aligned}
Z &=\operatorname{Tr}\left[e^{-\beta\hat{H}}\right] \\
&=\operatorname{Tr}\left[\left(e^{-\Delta\tau\hat{H}}\right)^{L_{\tau}}\right] \quad \text { where } e^{-\Delta\tau\hat{H}} \approx e^{-\Delta\tau\hat{H}_{I}} e^{-\Delta\tau\hat{H}_{0}}
\end{aligned}$$

Namely:

$$Z=\operatorname{Tr}\left[e^{-\Delta\tau\hat{H}_{I}} e^{-\Delta\tau\hat{H}_{0}} e^{-\Delta\tau\hat{H}_{I}} e^{-\Delta\tau\hat{H}_{0}} \cdots e^{-\Delta\tau\hat{H}_{I}} e^{-\Delta\tau\hat{H}_{0}}\right]$$

And from the previous text we already know, for the Hubbard U term, we have:

$$\begin{aligned}
e^{-\Delta\tau\hat{H}_{I}} &=\prod_{i} e^{-\Delta\tau U\left(\hat{n}_{i\uparrow}-\frac{1}{2}\right)\left(\hat{n}_{i\downarrow}-\frac{1}{2}\right)} \\
&=\prod_{i} \gamma \sum_{s_{i,l}=\pm 1} e^{\alpha s_{i,l}\left(\hat{n}_{i\uparrow}-\hat{n}_{i\downarrow}\right)} \\
&=\gamma^{N} \sum_{s_{i,l}=\pm 1}\left(\prod_{i} e^{\alpha s_{i,l}\hat{n}_{i\uparrow}} \prod_{i} e^{-\alpha s_{i,l}\hat{n}_{i\downarrow}}\right)
\end{aligned}$$

Where $\gamma=\frac{1}{2}e^{-\Delta\tau U/4}, \quad \cosh(\alpha)=e^{\Delta\tau U/2}$. $l$ is the label in the imaginary time direction. Here $N$ is the number of lattice sites $L \times L$.

For the sake of notational simplicity, we have the following notation: $\hat{H}_{0}=\boldsymbol{c}^{\dagger} T\boldsymbol{c}=-t \sum_{\langle ij\rangle\sigma} \hat{c}_{i\sigma}^{\dagger}\hat{c}_{j\sigma}+h.c.$, where $T$ is a matrix that only has matrix elements of $-t$ at nearest neighbours. In practice, a more convenient way to write this is to only write out the spin up or down part. The symbol $T$ hereafter in this text will only represent the partitioned small matrix within the large matrix, which is:

$$\boldsymbol{c}_{\uparrow}^{\dagger} T\boldsymbol{c}_{\uparrow}=-t \sum_{\langle ij\rangle} \hat{c}_{i\uparrow}^{\dagger}\hat{c}_{j\uparrow}+h.c. \quad \boldsymbol{c}_{\downarrow}^{\dagger} T\boldsymbol{c}_{\downarrow}=-t \sum_{\langle ij\rangle} \hat{c}_{i\downarrow}^{\dagger}\hat{c}_{j\downarrow}+h.c.$$

At this point, the partition function is written as:

$$Z =\sum_{s_{i,l}=\pm 1} \gamma^{NL_{\tau}} \operatorname{Tr}_{F}\left\{\prod_{l=1}^{L_{\tau}}\left[\left(\prod_{i} e^{\alpha s_{i,l}\hat{n}_{i\uparrow}}\right)\left(e^{-\Delta\tau c_{\uparrow}^{\dagger}Tc_{\uparrow}}\right)\left(\prod_{i} e^{-\alpha s_{i,l}\hat{n}_{i\downarrow}}\right)\left(e^{-\Delta\tau c_{\downarrow}^{\dagger}Tc_{\downarrow}}\right)\right]\right\}$$

It can clearly be seen that the form on the exponent of e is partitioned with respect to spin up and down, so it can be written in a more convenient form, and according to:

$$\operatorname{Tr}\left[e^{-\sum_{i,j} c_{i}^{\dagger}A_{i,j}c_{j}} e^{-\sum_{i,j} c_{i}^{\dagger}B_{i,j}c_{j}}\right]=\operatorname{Det}\left(1+e^{-\mathbf{A}}e^{-\mathbf{B}}\right)$$

we know that the result obtained by calculating the Trace over the fermion part is equivalent to the determinant of the exp of the corresponding matrix. At this time we have:

$$Z=\gamma^{NL_{\tau}} \sum_{\{s_{i,l}\}} \prod_{\sigma=\uparrow\downarrow} \operatorname{Det}\left[\mathbf{I}+\mathbf{B}^{\sigma}\left(L_{\tau}\Delta\tau,(L_{\tau}-1)\Delta\tau\right) \cdots \mathbf{B}^{\sigma}(\Delta\tau,0)\right]$$

Where:

$$\begin{array}{l}
\mathbf{B}^{\uparrow}\left(l_{2}\Delta\tau, l_{1}\Delta\tau\right)=\prod_{l=l_{1}+1}^{l_{2}} e^{\alpha\operatorname{Diag}(\vec{S}_{l})} e^{-\Delta\tau T} \\
\mathbf{B}^{\downarrow}\left(l_{2}\Delta\tau, l_{1}\Delta\tau\right)=\prod_{l=l_{1}+1}^{l_{2}} e^{-\alpha\operatorname{Diag}(\vec{S}_{l})} e^{-\Delta\tau T}
\end{array}$$

Where $\operatorname{Diag}(\vec{S}_{l})$ represents a matrix whose diagonal elements are respectively $s_{i,l}$.

Now we see that we have traced out the fermions, and the summation part of the partition function is merely a sum over an $L \times L \times L_{\tau}$ Ising auxiliary field. We temporarily set aside the matter of the auxiliary field; just like the Ising model, in the Ising model, when we are given a certain configuration of the Ising field, we obtain the weight (probability) corresponding to that configuration. Here, similarly, given the values of the $L \times L \times L_{\tau}$ Ising field, we have the corresponding weight:

$$W_{C}=\frac{\gamma^{NL_{\tau}} \prod_{\sigma=\uparrow\downarrow} \operatorname{Det}\left[\mathbf{I}+\mathbf{B}_{C}^{\sigma}\left(L_{\tau}\Delta\tau,(L_{\tau}-1)\Delta\tau\right) \cdots \mathbf{B}_{C}^{\sigma}(\Delta\tau,0)\right]}{Z}$$

Where adding $C$ refers to substituting the values of a specific Configuration to obtain the corresponding matrix before performing operations. For example, looking at the part $\mathbf{B}_{C}^{\uparrow}(\Delta\tau,0)=e^{\alpha\operatorname{Diag}(\vec{S}_{1})}e^{-\Delta\tau T}$, we find that the latter half is independent of the Ising field, while the former half depends on whether the value of $s_{i,1}$ is $+1$ or $-1$.

We know that when performing MCMC, what truly affects the computation is the ratio of weights, so we can omit identical constant factors and rewrite the weight as:

$$W_{C}=\prod_{\sigma=\uparrow\downarrow} \operatorname{Det}\left[\mathbf{I}+\mathbf{B}_{C}^{\sigma}\left(L_{\tau}\Delta\tau,(L_{\tau}-1)\Delta\tau\right) \cdots \mathbf{B}_{C}^{\sigma}(\Delta\tau,0)\right]=\prod_{\sigma=\uparrow\downarrow} \operatorname{Det}\left[\mathbf{I}+ \mathbf{B}_{C}^{\sigma}(\beta,0)\right]$$

Now we see that, given a specific auxiliary field: a configuration of the 2+1 Ising field, we can calculate the weight of that configuration (although it is very troublesome). We need to calculate each $\mathbf{B}$, which requires calculating the exp of a large matrix. The calculation of the exp of a matrix is usually done through diagonalisation, using what we learned in linear algebra: $e^{-A}=e^{-U^{\dagger}\operatorname{diag}(\lambda)U}=U^{\dagger}\operatorname{Diag}(e^{-\lambda_{i}})U$.

At this point, it is already a very strenuous operation, yet we still have to multiply the obtained $2L_{\tau}$ matrices together (which may lead to numerical errors during the multiplication process). After multiplying them, we must add an identity matrix, and then calculate the determinant of this massive $N \times N$ matrix. This is yet another strenuous operation. And after performing all these operations, we have now only calculated a single number: the weight corresponding to that configuration. And when you attempt to update to a new configuration and want to calculate the weight of the new configuration, thinking trivially, you would need to recalculate each $\mathbf{B}$ separately, multiply them together, and then calculate the determinant... This is obviously a terrifying amount of computation. Furthermore, directly multiplying $2L_{\tau}$ matrices that may have very large singularities will obviously easily cause numerical errors.

These problems will be solved one by one later, but at least now you can carry out the DQMC calculation of the Hubbard Model in an incredibly brute-force manner.